### Code para sacar mínimos locales: Delay electromecánico

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

# 1. Definir la ruta del archivo
file_path = (
    "C:/Users/adria/Downloads/Python-IMU/Adriana-Neuroengineering_lab/"
    "20260423/EMG_matlab/20260423_subject_01_cond_ASS-PRE-EXT-RIGHT_run03_ecr_fcr_final2.csv"
)

# Frecuencia de muestreo especificada (60 Hz)
FS = 60  

# 2. Cargar datos
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en {file_path}")
    exit()

ecr_signal = df["ECR_act"].values
wrist_signal = df["wrist_hand_r3_pos"].values
print(len(ecr_signal))
print(len(wrist_signal))

# 3. Detectar mínimos locales (invirtiendo la señal)
# A 60 Hz, una distancia de 3 muestras equivale a 50 ms. 
# Evita falsos positivos si hay pequeñas oscilaciones en muestras adyacentes.
min_distance_samples = 3 

min_indices_ecr, _ = find_peaks(-ecr_signal, distance=min_distance_samples)
min_indices_wrist, _ = find_peaks(-wrist_signal, distance=min_distance_samples)

# 4. Emparejamiento para el Retraso Electromecánico (EMD)
desfases_muestras = []

for idx_ecr in min_indices_ecr:
    # El mínimo mecánico de la muñeca debe ocurrir en el mismo instante o DESPUÉS de la activación
    futuros_wrist = min_indices_wrist[min_indices_wrist >= idx_ecr] #porque el movimiento siempre va despúes
    
    if len(futuros_wrist) > 0:
        # Tomamos el mínimo de muñeca más cercano en el tiempo
        siguiente_wrist = futuros_wrist[0] #0 por que es la mas cercana a la activation
        
        # Calcular desfase en muestras
        desfase = siguiente_wrist - idx_ecr
        
        # Umbral de ventana temporal: a 60 Hz, 18 muestras equivalen a 300 ms.
        # Si el desfase es mayor, probablemente pertenezca al siguiente ciclo de movimiento.
        if desfase <= 18: #18 porque ese desfase es como el tope para saber si ambos mínimos pertenecen al mismo ciclo de activación-movimiento 
            desfases_muestras.append(desfase)

# 5. Estadísticas finales y conversión a milisegundos
desfases_muestras = np.array(desfases_muestras)

if len(desfases_muestras) > 0:
    media_muestras = np.mean(desfases_muestras)
    std_muestras = np.std(desfases_muestras)
    
    # Conversión matemática a milisegundos para 60 Hz
    media_ms = (media_muestras / FS) * 1000
    std_ms = (std_muestras / FS) * 1000
    
    print("=" * 50)
    print("   RESULTADOS DEL RETRASO ELECTROMECÁNICO (EMD) a 60 Hz   ")
    print("=" * 50)
    print(f"Ciclos válidos emparejados: {len(desfases_muestras)}")
    print(f"Desfase medio: {media_muestras:.2f} muestras")
    print(f"Retraso Electromecánico Medio (EMD): {media_ms:.2f} ms")
    print(f"Desviación estándar: {std_ms:.2f} ms")
    print("=" * 50)
else:
    print("No se pudieron emparejar valles mecánicos coherentes tras las activaciones.")

1800
1800
   RESULTADOS DEL RETRASO ELECTROMECÁNICO (EMD) a 60 Hz   
Ciclos válidos emparejados: 190
Desfase medio: 3.74 muestras
Retraso Electromecánico Medio (EMD): 62.28 ms
Desviación estándar: 28.12 ms
